<a id="top"></a>
# House Sales: business & data understanding (esercitazione)

In questa esercitazione si lavora su un caso immobiliare basato su dati reali di compravendita di case nella King County (area di Seattle, USA), raccolti in circa un anno di osservazione. L’obiettivo è allenarsi a leggere la richiesta del business e a collegarla ai dati disponibili.

<a id="indice"></a>
## Indice

- [Contesto e obiettivo del progetto](#contesto-obiettivo)
- [Richieste del business](#richieste-business)
- [Da dove arrivano i dati](#origine-dati)
- [Caricamento del dataset House Sales](#caricamento-dataset)
- [Primo sguardo ai dati](#primo-sguardo)
- [Prime esplorazioni guidate](#prime-esplorazioni)
- [Punti chiave da portare avanti](#sintesi)


<a id="contesto-obiettivo"></a>
## Contesto e obiettivo del progetto

Immagina di lavorare nel team dati di una grande società immobiliare attiva nella contea di King. L’azienda compra e rivende case e offre anche un servizio di consulenza ai privati su come prezzare correttamente il proprio immobile.

Negli ultimi anni il management ha iniziato a percepire alcuni problemi ricorrenti:

- alcune case restano sul mercato per mesi perché il prezzo iniziale è troppo alto;
- altre vengono vendute velocemente, ma con la sensazione di aver “lasciato soldi sul tavolo”;
- in certe zone il mercato sembra più sensibile alla metratura, in altre alla qualità della casa o alla vicinanza a certe aree.

La direzione commerciale chiede al team dati di costruire uno strumento che aiuti a:

- stimare un intervallo di prezzo realistico per una casa prima di metterla sul mercato;
- capire quali caratteristiche spingono di più il prezzo verso l’alto o verso il basso;
- evidenziare casi potenzialmente sottostimati o sovrastimati rispetto al mercato della zona.

Dal punto di vista tecnico possiamo formulare il problema come un task di **regressione**: dato un insieme di caratteristiche di una casa, vogliamo prevedere il suo **prezzo di vendita**.



<a id="richieste-business"></a>
## Richieste del business

Per lavorare bene serve esplicitare le richieste, anche quando arrivano in modo un po’ informale. In questo caso possiamo riassumerle in alcune domande guida:

1. **Prezzo atteso per una nuova casa**  
   Dato un immobile con certe caratteristiche (metratura, numero di stanze, zona, condizioni), quale fascia di prezzo è plausibile?

2. **Fattori che guidano il prezzo**  
   Quanto pesano davvero le diverse caratteristiche sul prezzo finale? Ad esempio: contano di più i metri quadri, la posizione o lo stato dell’immobile?


Nel resto dell’esercitazione userai il dataset House Sales per capire **se** e **in che misura** queste domande possono trovare risposta con i dati a disposizione.


<a id="origine-dati"></a>
## Da dove arrivano i dati

Il dataset `house_sales` raccoglie informazioni su migliaia di compravendite di case nella **King County** (l’area di Seattle, nello stato di Washington, USA). I dati coprono circa un anno di transazioni, tra il 2014 e il 2015.

Ogni riga rappresenta una casa venduta; le colonne descrivono:

- il **prezzo di vendita** (`price`);
- caratteristiche fisiche dell’immobile, come:
  - `bedrooms`, `bathrooms`;
  - `sqft_living`, `sqft_lot` (metratura interna e del lotto);
  - `floors`, `sqft_above`, `sqft_basement`;
- indici qualitativi come `condition` e `grade`, che riassumono la qualità percepita dell’abitazione;
- informazioni sulla **posizione**:
  - `zipcode` (zona),
  - `lat`, `long` (coordinate geografiche);
- alcune variabili di contesto, ad esempio:
  - la **data di vendita** (`date`);
  - misure di metratura della casa e del lotto nelle vicinanze (`sqft_living15`, `sqft_lot15`).

Tutti questi elementi sono potenziali candidati come feature per il modello, ma in questa fase li guardiamo ancora solo come **pezzi di informazione** a disposizione.

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Attenzione**

Il dataset rappresenta il mercato immobiliare **in una singola area geografica** e in un **periodo di tempo limitato**. Le relazioni che troverai (per esempio tra metratura e prezzo) non vanno automaticamente estese ad altre città o ad altri anni.

</div>


<a id="caricamento-dataset"></a>
## Caricamento del dataset House Sales

Per lavorare con i dati useremo `pandas` come libreria principale. In questo esercizio il dataset viene scaricato da OpenML tramite `scikit-learn`, così non serve gestire file manualmente.

- `fetch_openml` scarica il dataset da OpenML e lo restituisce come oggetto simile a un dizionario con dati e metadati.  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html
- `pandas.DataFrame` è la struttura tabellare base di pandas.  
  Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html

Quando si usa `as_frame=True`, l’oggetto restituito da `fetch_openml` contiene un attributo `.frame` che è un `DataFrame` con tutte le colonne (feature e target) già unite.

L’obiettivo di questa sezione è:

- ottenere un `DataFrame` con tutte le colonne (feature + target);
- verificare rapidamente quante righe e quante colonne ci sono.


In [1]:
import pandas as pd
from sklearn.datasets import fetch_openml  
import plotly.express as px

# Scarica il dataset House Sales da OpenML
raw = fetch_openml(data_id=42092, as_frame=True)  
# Estrai il DataFrame con tutte le colonne
house = raw.frame.copy()  

# Controlla dimensioni: numero di righe e colonne
house.shape


(21613, 20)

In [2]:
house

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,20140521T000000,360000.0,3,2.50,1530,1131,3.0,0,0,3,8,1530,0,2009,0,98103,47.6993,-122.346,1530,1509
21609,20150223T000000,400000.0,4,2.50,2310,5813,2.0,0,0,3,8,2310,0,2014,0,98146,47.5107,-122.362,1830,7200
21610,20140623T000000,402101.0,2,0.75,1020,1350,2.0,0,0,3,7,1020,0,2009,0,98144,47.5944,-122.299,1020,2007
21611,20150116T000000,400000.0,3,2.50,1600,2388,2.0,0,0,3,8,1600,0,2004,0,98027,47.5345,-122.069,1410,1287


<a id="sintesi"></a>
## Punti chiave da portare avanti

Prima di passare alla modellazione, conviene fissare alcuni elementi essenziali emersi da questa esercitazione:

- il problema di business è naturalmente formulabile come **previsione di un prezzo continuo** (task di regressione);
- il dataset contiene migliaia di transazioni reali, con un insieme ricco di feature sulla casa e sulla posizione;
- la distribuzione dei prezzi tende a essere molto asimmetrica, con una coda di case molto costose;
- alcune variabili, come il numero di camere o la metratura interna, mostrano da subito una relazione chiara con il prezzo.

Nel proseguimento del corso lo scopo sarà usare queste osservazioni per:

- pulire e preparare i dati in modo coerente con il problema;
- progettare un modello che dia risposte utili alle domande del business, non solo buone metriche tecniche.
